In [2]:
import argparse
import os
import sys
import cv2
import json
from pathlib import Path
# from labels import CLASS_TO_IDX, load_game_labels
from SoccerNet.Downloader import SoccerNetDownloader

c:\Users\parth\.pyenv-win-venv\envs\capdis_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def _get_downloader(local_dir: str, password: str = None):
    downloader = SoccerNetDownloader(local_dir)
    if password:
        downloader.password = password
    return downloader


In [ ]:
dl = _get_downloader('./data/labels', "s0cc3rn3t")
dl.downloadGames(files=["Labels-v2.json"], split=["train", "valid", "test"])

In [5]:
dl = _get_downloader('./data/resnet_features', "s0cc3rn3t")
dl.downloadGames(
        files=["1_ResNET_TF2_PCA512.npy", "2_ResNET_TF2_PCA512.npy"],
        split=["train", "valid", "test"],
    )

KeyboardInterrupt: 

In [ ]:
dl = _get_downloader('./data/mask_rcnn_boxes', "s0cc3rn3t")
dl.downloadGames(
        files=[
            "1_player_boundingbox_maskrcnn.json",
            "2_player_boundingbox_maskrcnn.json",
        ],
        split=["train", "valid", "test"],
    )

In [29]:
from SoccerNet.utils import getListGames
import random
all_games = getListGames(split=["train"])
games = random.sample(all_games, 15)
games

['germany_bundesliga\\2016-2017\\2016-12-20 - 22-00 Dortmund 1 - 1 FC Augsburg',
 'italy_serie-a\\2014-2015\\2015-04-25 - 21-45 Inter 2 - 1 AS Roma',
 'europe_uefa-champions-league\\2014-2015\\2014-11-05 - 22-45 Bayern Munich 2 - 0 AS Roma',
 'europe_uefa-champions-league\\2015-2016\\2015-11-24 - 22-45 Bayern Munich 4 - 0 Olympiakos Piraeus',
 'germany_bundesliga\\2016-2017\\2016-09-20 - 21-00 Wolfsburg 1 - 5 Dortmund',
 'england_epl\\2015-2016\\2016-03-19 - 18-00 Chelsea 2 - 2 West Ham',
 'italy_serie-a\\2016-2017\\2016-09-10 - 21-45 Palermo 0 - 3 Napoli',
 'germany_bundesliga\\2015-2016\\2015-08-30 - 16-30 Dortmund 3 - 1 Hertha Berlin',
 'italy_serie-a\\2016-2017\\2016-11-06 - 17-00 Palermo 1 - 2 AC Milan',
 'europe_uefa-champions-league\\2015-2016\\2015-11-25 - 22-45 Atl. Madrid 2 - 0 Galatasaray',
 'italy_serie-a\\2016-2017\\2017-02-12 - 14-30 Crotone 0 - 2 AS Roma',
 'europe_uefa-champions-league\\2015-2016\\2015-09-15 - 21-45 Galatasaray 0 - 2 Atl. Madrid',
 'italy_serie-a\\2016-

In [ ]:


for i, game_path in enumerate(games):
    game_dir = os.path.join("./yolo_frames", game_path)
    os.makedirs(game_dir, exist_ok=True)
    print(f"\n  [{i+1}/15] {Path(game_path).name}")

    try:
        dl.downloadGame(
            game=game_path,
            files=["1_224p.mkv", "2_224p.mkv"],
        )
    except Exception as e:
        print(f"    WARN: download failed — {e}")
        continue

    half_events = load_game_labels(game_dir)

    for half in (1,2):
        video_path = os.path.join(game_dir, f"{half}_224p.mkv")
        if not os.path.exists(video_path):
            continue
        n_saved = _extract_frames_around_events(
            video_path=video_path,
            events=half_events.get(half, []),
            out_dir=os.path.join('./yolo_frames', game_path, "frames"),
            sample_fps=5.0,
            context_secs=3.0,
            game_tag=f"g{i:02d}_h{half}",
        )

        os.remove(video_path)

        print(f"Half {half}: {n_saved} frames saved")


  [1/15] 2016-01-13 - 19-45 Chelsea 2 - 0 Newcastle United
HTTP Error 404: Not Found
HTTP Error 404: Not Found

  [2/15] 2016-03-02 - 19-45 Arsenal 2 - 1 Tottenham Hotspur
HTTP Error 404: Not Found
HTTP Error 404: Not Found

  [3/15] 2015-05-24 - 16-00 Manchester United 1 - 1 Arsenal
HTTP Error 404: Not Found


KeyboardInterrupt: 

In [ ]:
dl.downloadGame